# Notebook 1: Pre-compute Event Embeddings with Qwen3-4B

In [ ]:
!pip install -q transformers accelerate torch

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_token = user_secrets.get_secret("HF_TOKEN")

login(token=HF_token)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## Load Data

In [ ]:
DATA_DIR = "/kaggle/input/finance-world-model-dataset"

events_df = pd.read_csv(f"{DATA_DIR}/gdelt_events_with_market_impact.csv")
events_df["event_time"] = pd.to_datetime(events_df["event_time"])
events_df = events_df[events_df["market_impact"] == 1].reset_index(drop=True)
print(f"Market-relevant events: {len(events_df)}")

market_df = pd.read_csv(f"{DATA_DIR}/combined_commodity_data.csv", parse_dates=["date"], index_col="date")
print(f"Market data: {market_df.shape}")
events_df.head(3)

## Format Event Texts with Market Context

In [ ]:
# Features to include in market context prefix
CONTEXT_MAP = {
    "YF_Price_Crude Oil (WTI)": ("WTI", "${:.2f}"),
    "YF_Price_Crude Oil (Brent)": ("Brent", "${:.2f}"),
    "YF_Price_Gold": ("Gold", "${:.0f}"),
    "YF_Price_Natural Gas": ("NatGas", "${:.2f}"),
    "FRED_DGS10": ("10Y", "{:.2f}%"),
    "FRED_DTWEXBGS": ("DXY", "{:.1f}"),
    "FRED_SP500": ("S&P500", "{:.0f}"),
    "FRED_VIXCLS": ("VIX", "{:.1f}"),
}


def format_event_text(event_row, market_df):
    """Build event text with prepended market context from the event's date."""
    event_date = event_row["event_time"]
    summary = event_row["event_summary"]

    # Find nearest available market date (event date or earlier)
    available = market_df.index[market_df.index <= event_date]
    if len(available) == 0:
        available = market_df.index[:1]  # fallback to earliest
    market_date = available[-1]
    market_row = market_df.loc[market_date]

    # Build context string
    parts = []
    for col, (label, fmt) in CONTEXT_MAP.items():
        val = market_row[col]
        parts.append(f"{label} {fmt.format(val)}")
    context = ", ".join(parts)

    date_str = event_date.strftime("%Y-%m-%d")
    return f"Market on {date_str}: {context}\nEvent ({date_str}): {summary}"


# Format all events
event_texts = []
for _, row in events_df.iterrows():
    event_texts.append(format_event_text(row, market_df))

print(f"Formatted {len(event_texts)} event texts")
print(f"\n--- Example ---\n{event_texts[0][:300]}...")

## Load Qwen3-4B

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print(f"Model loaded. Hidden size: {model.config.hidden_size}")

## Extract Embeddings

In [ ]:
BATCH_SIZE = 16
embeddings = {}

for start in range(0, len(event_texts), BATCH_SIZE):
    batch_texts = event_texts[start : start + BATCH_SIZE]
    batch_indices = list(range(start, start + len(batch_texts)))

    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # Last hidden state: (batch, seq_len, hidden_size)
    last_hidden = outputs.hidden_states[-1]

    # Extract last non-pad token's hidden state for each sample
    attention_mask = inputs["attention_mask"]
    for i, idx in enumerate(batch_indices):
        seq_len = attention_mask[i].sum().item()
        # Last real token embedding
        emb = last_hidden[i, seq_len - 1, :].cpu().float()
        embeddings[idx] = emb

    if (start // BATCH_SIZE) % 10 == 0:
        print(f"Processed {start + len(batch_texts)}/{len(event_texts)} events")

print(f"\nDone. Extracted {len(embeddings)} embeddings.")
print(f"Embedding dim: {next(iter(embeddings.values())).shape}")

## Save Embeddings

In [ ]:
# Save as a single tensor + index mapping for efficiency
output = {
    "embeddings": embeddings,           # dict: int -> tensor(hidden_size)
    "hidden_size": model.config.hidden_size,
    "model_id": MODEL_ID,
    "n_events": len(embeddings),
}

torch.save(output, "event_embeddings.pt")
print(f"Saved event_embeddings.pt")

# Also save the event texts for reference in Stage 2
events_df["formatted_text"] = event_texts
events_df.to_csv("events_with_formatted_text.csv", index=False)
print(f"Saved events_with_formatted_text.csv")